# **MicroCTD Data preprocessing** 

<span style="color:red">This code was last tested on February 13, 2026, and is running without issues.</span>


**PURPOSE:** Open MicroCTD Data and perform the **first step** for preprocessing it in Python before synchronizing with the ADCP data.

This code will generate the pickle files: **MCTD_time** and **MCTD_data**, as well as the corresponding files for each of the 3 distinct parts of the campaign: **drift**, **jetties**, and **transect**.

__________________________________________________________________________________________  

**Input files:**  

DAT_001 a DAT_016  

on path: = "/Users/deboragadelha/Jupyter/barros_et_al_2026/input_dir/MCTD_profiles_data/"

**Output files:** 

MCTD_time.pkl  
MCTD_time_drift.pkl, MCTD_time_jetties.pkl and MCTD_time_transect.pkl 
 
MCTD_data.pkl  
MCTD_data_drift.pkl, MCTD_data_jetties.pkl, and MCTD_data_transect.pkl 

on path: = "/Users/deboragadelha/Jupyter/barros_et_al_2026/output_dir/processed_data/"  
__________________________________________________________________________________________

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dateutil.parser import parse
import rasterio
import rasterio.plot as rplot
from IPython import display
import time as pytime
import pickle
import datetime
import scipy.io as scio
import os
os.getcwd()

from datetime import timedelta

# scipy.io.loadmat

Within this fieldwork, three distinct surveys were conducted. First, a drift was carried out along the thalweg, from inside the inlet to offshore. Then, MicroCTD data were collected at three different locations within the plume, near the jetties. Lastly, a transect was performed in front of the pilot station, with five sampling stations.

## Identifying each file corresponding to the three distinct internal surveys:

DAT_001 a DAT_008: drift along the thalweg - Drift data

DAT_009: Station 1 near the jetties (plume)  - Jetties data  
DAT_010: Station 2 near the jetties (plume)  - Jetties data  
DAT_011: Station 3 near the jetties (plume)  - Jetties data  

DAT_012: Station 1 of the transect  - Transect data  
DAT_013: Station 2 of the transect  - Transect data  
DAT_014: Station 3 of the transect  - Transect data  
DAT_015: Station 4 of the transect  - Transect data  
DAT_016: Station 5 of the transect  - Transect data

In [2]:
path = "/Users/deboragadelha/Jupyter/barros_et_al_2026/input_dir/MCTD_profiles_data/"
dirfiles = sorted(os.listdir(path))  # <-- here we put on alphabetical order

# create a list of processed profiles
profile_files = []
c = 0
for f in dirfiles:
    if ('profile' in f) and f.endswith('.mat'):
        profile_files.append(f)
        print(c, f)
        c += 1

0 DAT_001_profile_001_20220718_022104.mat
1 DAT_001_profile_002_20220718_022234.mat
2 DAT_001_profile_003_20220718_022622.mat
3 DAT_001_profile_004_20220718_022730.mat
4 DAT_001_profile_005_20220718_022836.mat
5 DAT_001_profile_006_20220718_022941.mat
6 DAT_001_profile_007_20220718_023053.mat
7 DAT_001_profile_008_20220718_023210.mat
8 DAT_001_profile_009_20220718_023324.mat
9 DAT_002_profile_001_20220718_023725.mat
10 DAT_002_profile_002_20220718_024103.mat
11 DAT_002_profile_003_20220718_024215.mat
12 DAT_002_profile_004_20220718_024316.mat
13 DAT_002_profile_005_20220718_024421.mat
14 DAT_002_profile_006_20220718_024523.mat
15 DAT_002_profile_007_20220718_024623.mat
16 DAT_002_profile_008_20220718_024718.mat
17 DAT_003_profile_001_20220718_025603.mat
18 DAT_003_profile_002_20220718_025702.mat
19 DAT_003_profile_003_20220718_025800.mat
20 DAT_003_profile_004_20220718_025901.mat
21 DAT_003_profile_005_20220718_025959.mat
22 DAT_003_profile_006_20220718_030057.mat
23 DAT_003_profile_00

In [3]:
# load the files (could be done in the previous for, but the idea is to check the files first!

profiles_mat = []
for f in profile_files:  # to get only the cross section
    mat = scio.loadmat(path + f)
    profiles_mat.append(mat)

In [4]:
#Listing the variables in the file:

keys = mat.keys()

for i,k in enumerate(keys):
    print(i, k)

0 __header__
1 __version__
2 __globals__
3 junta


## Slicing variables, correcting MCTD_time and preparing a matrix of MCTD_data

In [5]:
profile_data = []
profile_time = []

for mat in profiles_mat:
    # Unpack the structure from the .mat file
    dados = mat['junta']
    time = dados['file_datetime'][0][0][0]
    dissipation = dados['dissipation'][0][0]
    data_slow = dados['Data_slow'][0][0]
    data_fast = dados['Data_fast'][0][0]
    hdr_slow = dados['slow_list'][0][0]
    hdr_fast = dados['fast_list'][0][0]

    # Extract variable names as strings
    hdr_slow = [x[0] for x in hdr_slow[0][:]]
    hdr_fast = [x[0] for x in hdr_fast[0][:]]

    # Convert start time of profile to datetime object
    time_dt = parse(time)

    # Apply 9-hour correction to fix known MCTD clock offset
    time_dt_corrected = time_dt + datetime.timedelta(hours=9)
    #Following the field campaign, we noticed that the equipment's internal clock was 
    #set incorrectly (9 hours behind), so we had to manually adjust the timestamps.

    # Compute exact profile time by adding seconds since deployment start
    time_since_start = data_slow[28, 0]
    time_profile = time_dt_corrected + datetime.timedelta(seconds=time_since_start)

    # Convert profile time to Julian format and repeat for all samples
    time_profile_n = mdates.date2num(time_profile)
    time_profile_nn = np.full(data_slow.shape[1], time_profile_n)

    # Extract variables of interest
    pressure = data_slow[11,:]
    salinity = data_slow[12,:]
    temperature = data_slow[17,:]
    density = data_slow[26,:] + 1000 # absolute density
    fall_velocity = data_slow[19,:]
    inclination = data_slow[4,:]
    JAC_T = data_slow[7,:]#temperature from JAC sensor
    JAC_C = data_slow[6,:]#conductivity from JAC sensor
    dissipation1 = dissipation[0,:]
    dissipation2 = dissipation[1,:]

    # Build the data matrix for the profile (each row = one measurement)
    j_profile_data = np.vstack((
        time_profile_nn,
        pressure,
        salinity,
        temperature,
        density,
        dissipation1,
        dissipation2,
        fall_velocity,
        inclination,
        JAC_T,
        JAC_C
    )).T

    # Filter out samples with poor inclination (keep only vertical casts)
    j_profile_data = j_profile_data[j_profile_data[:, 7] > 0.6, :]

    # Store the cleaned profile and its associated timestamp
    profile_data.append(j_profile_data)
    profile_time.append(time_profile)


The slicing of the MCTD data can be done based on the time variable. However, since I already know which file corresponds to each part of the fieldwork, I'm doing it here directly.

In [6]:
# Slicing MCTD_time and profile_data in 3 different variables: drift, jetties and transect

MCTD_time = profile_time
MCTD_time_drift = profile_time[0:109]
MCTD_time_jetties = profile_time[109:126]
MCTD_time_transect = profile_time[126:]

MCTD_data = profile_data
MCTD_data_drift = profile_data[0:109]
MCTD_data_jetties = profile_data[109:126]
MCTD_data_transect = profile_data[126:]

In [7]:
#checking it out
print(len(MCTD_time))     # Deve ser 152
print(len(MCTD_time_drift))     # Deve ser 108
print(len(MCTD_time_jetties))   # Deve ser 17
print(len(MCTD_time_transect))  # Deve ser 26

print(len(MCTD_data))     # Deve ser 152
print(len(MCTD_data_drift))     # Deve ser 108
print(len(MCTD_data_jetties))   # Deve ser 17
print(len(MCTD_data_transect))  # Deve ser 26

152
109
17
26
152
109
17
26


In [8]:
# path to the folder where the pickles will be saved:

output_dir = '../output_dir/processed_data'
os.makedirs(output_dir, exist_ok=True)


# Save the pickle on the right folder:


## ====== MCTD time =======

with open(os.path.join(output_dir, 'MCTD_time.pkl'), 'wb') as io:
    pickle.dump(MCTD_time, io)

with open(os.path.join(output_dir, 'MCTD_time_drift.pkl'), 'wb') as io:
    pickle.dump(MCTD_time_drift, io)

with open(os.path.join(output_dir, 'MCTD_time_jetties.pkl'), 'wb') as io:
    pickle.dump(MCTD_time_jetties, io)

with open(os.path.join(output_dir, 'MCTD_time_transect.pkl'), 'wb') as io:
    pickle.dump(MCTD_time_transect, io)


## ====== MCTD data =======

with open(os.path.join(output_dir, 'MCTD_data.pkl'), 'wb') as io:
    pickle.dump(MCTD_data, io)

with open(os.path.join(output_dir, 'MCTD_data_drift.pkl'), 'wb') as io:
    pickle.dump(MCTD_data_drift, io)

with open(os.path.join(output_dir, 'MCTD_data_jetties.pkl'), 'wb') as io:
    pickle.dump(MCTD_data_jetties, io)

with open(os.path.join(output_dir, 'MCTD_data_transect.pkl'), 'wb') as io:
    pickle.dump(MCTD_data_transect, io)

<span style="color:red">This code was last tested on February 13, 2026, and is running without issues.</span>


## END OF CODE